# 01. Blocking receive와 passive awareness

목표: 메시지를 듣는 데 foreground step을 쓰는 방식과 background watcher가 다음 step 경계에 메시지를 노출하는 방식을 작은 이산 사건 모형으로 비교합니다. Python 표준 라이브러리만 사용하며 원 논문 시스템을 그대로 재현하지 않는 toy simulation입니다. 위에서 아래로 실행하세요.

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class Message:
    sent_before_step: int
    sender: str
    content: str

messages = [
    Message(2, 'agent-2', '설정 스위치를 찾았습니다.'),
    Message(4, 'agent-3', '초기 가설과 충돌하는 로그입니다.'),
]
print(messages)

## 두 수신 모드

blocking은 정해진 step을 `listen`에 사용합니다. passive는 매 작업 step이 시작될 때 이미 도착한 메시지를 watcher buffer에서 가져옵니다. `work_done`은 실제 조사에 쓴 step 수입니다.

In [ ]:
def run(mode: str, total_steps: int = 6, listen_steps=(3, 6)):
    visible, work_done, timeline = [], 0, []
    for step in range(1, total_steps + 1):
        arrived = [m for m in messages if m.sent_before_step < step and m not in visible]
        if mode == 'passive':
            visible.extend(arrived)  # watcher가 step 경계에서 노출
            action = 'work'
            work_done += 1
        elif step in listen_steps:
            visible.extend(arrived)  # foreground step을 들어야만 수신
            action = 'listen'
        else:
            action = 'work'
            work_done += 1
        timeline.append((step, action, work_done, len(visible)))
    return timeline

for mode in ('blocking', 'passive'):
    print(f'\n{mode}')
    print('step action work_done visible_messages')
    for row in run(mode):
        print(*row)

In [ ]:
blocking = run('blocking')
passive = run('passive')
assert blocking[-1][2] == 4
assert passive[-1][2] == 6
assert passive[-1][3] == 2
print('같은 6 step 예산에서 passive는 듣기 전용 step 없이 작업 2회를 더 수행했습니다.')

## 생각해 볼 점

이 모형은 통신 내용의 token 비용과 메시지로 인한 attention disruption을 생략합니다. `messages`를 늘리고, 메시지 하나를 읽을 때 작업 효율이 낮아지는 penalty를 추가해 언제 passive 방식의 순이득이 사라지는지 실험해 보세요.